# 🎵 Moosic — Automated Playlist Creation

**Objective:** Use Spotify audio features and K-Means clustering to automatically generate playlists of 50–250 songs each.

**Business context:** Moosic is a startup that creates curated playlists by music experts. As the business scales, they want to explore whether Machine Learning can automate — or assist — the playlist creation process.

**Method:** A 2-step approach:
1. **K-Means clustering** (K=8) groups songs by overall musical similarity using 9 audio features
2. **Mood-based splitting** divides each cluster into smaller playlists using a mood score derived from Russell's Circumplex Model of Affect

**Result:** 27 playlists, each containing 152–213 songs ✅

---
## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import silhouette_score
from matplotlib.colors import LinearSegmentedColormap

pd.set_option("display.max_columns", None)

In [ ]:
# Load data
songs_df = pd.read_csv("3_spotify_5000_songs.csv")

# Clean column names (trailing spaces in original CSV)
songs_df.columns = songs_df.columns.str.strip()

print(f"Dataset: {songs_df.shape[0]} songs, {songs_df.shape[1]} columns")
songs_df.head()

---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
songs_df.info()

In [ ]:
songs_df.describe()

In [ ]:
# Check missing values
missing = songs_df.isnull().sum()
missing[missing > 0]

In [ ]:
# Check duplicates
duplicates = songs_df.duplicated(subset=["name", "artist"]).sum()
print(f"Duplicate songs: {duplicates}")

In [ ]:
# Distribution of audio features
audio_features = ["danceability", "energy", "loudness", "speechiness",
                  "acousticness", "instrumentalness", "liveness",
                  "valence", "tempo"]

# Colors by feature group (correlated features share the same color):
# Pink: rhythm & mood (danceability, valence, tempo)
# Teal: intensity (energy, loudness)
# Yellow: voice & texture (speechiness, acousticness, instrumentalness)
# Orange: performance (liveness)
feature_colors = {
    "danceability": "#FF69B4", "energy": "#00CED1", "loudness": "#00CED1",
    "speechiness": "#FFD700", "acousticness": "#FFD700", "instrumentalness": "#FFD700",
    "liveness": "#FF8C00", "valence": "#FF69B4", "tempo": "#FF69B4",
}

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i, feat in enumerate(audio_features):
    ax = axes[i // 3, i % 3]
    songs_df[feat].hist(bins=30, ax=ax, color=feature_colors[feat], edgecolor="white", alpha=0.85)
    ax.set_title(feat, fontsize=12, fontweight="bold")
    ax.set_ylabel("")
plt.suptitle("Distribution of Audio Features\n(same color = correlated features)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Observations:**
- Which features have a uniform distribution?
- Which are skewed (e.g. instrumentalness, speechiness)?
- Are there outliers?

In [ ]:
# Correlation heatmap with Moosic brand colors
moosic_colors = ["#00CED1", "#FFD700", "#FF69B4"]
moosic_cmap = LinearSegmentedColormap.from_list("moosic", moosic_colors)

plt.figure(figsize=(10, 8))
sns.heatmap(songs_df[audio_features].corr(), annot=True, cmap=moosic_cmap,
           center=0, fmt=".2f", square=True,
           annot_kws={"fontsize": 9, "fontweight": "bold"})
plt.title("Correlation Heatmap — Audio Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Key correlations:**
- `energy` ↔ `loudness`: strong positive — loud songs tend to be energetic
- `energy` ↔ `acousticness`: strong negative — acoustic songs tend to be low energy
- If two features are highly correlated, they carry redundant information

---
## 3. Feature Selection

**Kept (9 features):** Audio characteristics that describe the "mood" and "feel" of a song — danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo.

**Dropped:**
- `name`, `artist`, `id`, `html` → text, not numeric — K-Means requires numerical input
- `type` → same value for all rows (no discriminating power)
- `duration_ms` → song length doesn't define mood
- `key`, `mode`, `time_signature` → categorical variables — Euclidean distance doesn't make sense for categories (the "distance" between key C and key D is meaningless)

In [ ]:
# Select audio features
features_to_use = ["danceability", "energy", "loudness", "speechiness",
                   "acousticness", "instrumentalness", "liveness",
                   "valence", "tempo"]

features_df = songs_df[features_to_use].copy()

print(f"Features selected: {len(features_to_use)}")
print(f"\nFeature ranges (before scaling):")
for col in features_to_use:
    print(f"  {col:20s}  {features_df[col].min():.2f} to {features_df[col].max():.2f}")

---
## 4. Feature Scaling

K-Means calculates distances between data points. Features with larger ranges dominate the distance calculation:
- `tempo`: 0–250 BPM → a 10% change = distance increase of 25
- `energy`: 0–1 → a 10% change = distance increase of 0.1

Without scaling, the algorithm would cluster songs primarily by tempo and loudness, effectively ignoring features like energy or valence.

**MinMaxScaler** brings all features to the 0–1 range, ensuring each feature contributes equally to the distance calculation.

In [ ]:
# MinMaxScaler — brings everything to 0-1
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features_df)

# Back to DataFrame
scaled_df = pd.DataFrame(features_scaled,
                         index=features_df.index,
                         columns=features_df.columns)

print("After scaling — all features now 0-1:")
scaled_df.describe().loc[["min", "max"]]

---
## 5. Choosing K — How Many Clusters?

### 5.1 Elbow Method
Run K-Means for K=1 to 50 and plot inertia (sum of squared distances to centroids). Look for the "elbow" — the point where improvement slows down dramatically.

In [ ]:
# Elbow method
inertias = []
K_range = range(1, 51)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    kmeans.fit(scaled_df)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, "o-", color="#2a9d8f", linewidth=2, markersize=8)
plt.xlabel("Number of Clusters (K)", fontsize=12)
plt.ylabel("Inertia", fontsize=12)
plt.title("Elbow Method", fontsize=14, fontweight="bold")
plt.xticks(range(0, 51, 5))
plt.grid(True, alpha=0.3)
plt.show()

### 5.2 Silhouette Score
Measures how well each point fits in its cluster vs neighboring clusters. Ranges from -1 (wrong cluster) to +1 (perfect fit). We look for local maxima.

In [ ]:
# Silhouette scores
silhouette_scores = []
K_range_sil = range(2, 51)

for k in K_range_sil:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(scaled_df)
    score = silhouette_score(scaled_df, labels)
    silhouette_scores.append(score)
    print(f"K={k:2d}  Silhouette Score: {score:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(K_range_sil, silhouette_scores, "o-", color="#ec9750", linewidth=2, markersize=8)
plt.xlabel("Number of Clusters (K)", fontsize=12)
plt.ylabel("Silhouette Score", fontsize=12)
plt.title("Silhouette Score per K", fontsize=14, fontweight="bold")
plt.xticks(range(0, 51, 5))
plt.grid(True, alpha=0.3)
plt.show()

### 5.3 Business Considerations

**Mathematical criteria** suggest a small K (around 4–8) for optimal cluster quality.

**Business requirement:** Moosic wants playlists with **50–250 songs** each.
- With ~5,235 songs: minimum K = 5235 / 250 ≈ **21 playlists**
- With ~5,235 songs: maximum K = 5235 / 50 ≈ **105 playlists**

**The conflict:** K=8 gives a good silhouette score (0.2778) but produces clusters of ~600 songs — far above the 250 limit.

**Our solution (instructor recommendation):** A **2-step approach:**
1. K-Means with K=8 for mathematically strong, holistic clusters
2. Split each cluster into sub-playlists by mood score to meet the 50–250 constraint

This gives us the best of both worlds: good clustering quality AND business-appropriate playlist sizes.

---
## 6. Step 1 — K-Means Clustering

In [ ]:
# Step 1: Broad holistic clusters
# Small K gives mathematically better clusters
# Step 2 will split these into 50-250 song playlists using mood scores
chosen_k = 8

kmeans_final = KMeans(n_clusters=chosen_k, random_state=42, n_init="auto")
kmeans_final.fit(scaled_df)

# Add clusters
songs_df["cluster"] = kmeans_final.labels_
scaled_df["cluster"] = kmeans_final.labels_

# Songs per cluster
print("Step 1 — Broad clusters (too large for playlists):")
print(songs_df["cluster"].value_counts().sort_index())
print(f"\nInertia: {kmeans_final.inertia_:.2f}")
print(f"Silhouette Score: {silhouette_score(scaled_df[features_to_use], kmeans_final.labels_):.4f}")
print(f"\nNote: These clusters average ~{len(songs_df)//chosen_k} songs — above the 250 limit.")
print("Step 2 will split them by mood score.")

In [ ]:
# Cluster size check (Step 1)
cluster_sizes = songs_df["cluster"].value_counts().sort_index()

print("Step 1 — Cluster size check:")
print(f"  Smallest cluster: {cluster_sizes.min()} songs")
print(f"  Largest cluster:  {cluster_sizes.max()} songs")
print(f"  Average cluster:  {cluster_sizes.mean():.0f} songs")
print(f"\nAll clusters exceed 250 — Step 2 needed.")

---
## 7. Cluster Analysis

### 7.1 Feature Averages per Cluster
What "character" does each cluster have?

In [ ]:
# Averages per cluster
cluster_means = scaled_df.groupby("cluster")[features_to_use].mean()
cluster_means

In [ ]:
# Heatmap of cluster profiles
plt.figure(figsize=(14, 6))
sns.heatmap(cluster_means, annot=True, cmap="YlOrRd", fmt=".2f",
           xticklabels=features_to_use,
           yticklabels=[f"Cluster {i}" for i in range(chosen_k)])
plt.title("Cluster Profiles — Average Features per Cluster", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 7.2 Radar Chart
Visual cluster profiles — effective for non-technical audiences.

In [ ]:
# Radar chart (Plotly)
scatter_objects = []
categories = features_to_use
cluster_feat_means = scaled_df.groupby("cluster")[features_to_use].mean()

colors = ["#2a9d8f", "#ec9750", "#e76f51", "#264653", "#e9c46a",
          "#6c9eff", "#b68aff", "#ff6b9d", "#5cdb95", "#ffd76d"]

for cluster in sorted(scaled_df["cluster"].unique()):
    cluster_means_row = cluster_feat_means.loc[cluster]
    cluster_scatter = go.Scatterpolar(
        r=cluster_means_row,
        theta=categories,
        fill="toself",
        name=f"Cluster {cluster}",
        line=dict(color=colors[cluster % len(colors)])
    )
    scatter_objects.append(cluster_scatter)

fig = go.Figure()
fig.add_traces(scatter_objects)
fig.update_layout(
    title_text="Radar Chart — Audio Feature Profile per Cluster",
    height=600, width=800,
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True
)
fig.show()

In [ ]:
# Feature variance across clusters — which features differentiate clusters most?
cluster_feat_means.var().sort_values(ascending=False).plot(kind='bar', figsize=(10, 5))
plt.ylabel("Variance of cluster means")
plt.title("Which Features Differentiate Clusters Most?", fontweight="bold")
plt.show()

---
## 8. Step 2 — Mood-Based Playlist Splitting

### 8.1 Theoretical Foundation: Russell's Circumplex Model of Affect

Our mood score is inspired by **Russell's Circumplex Model** (1980), a well-established psychological framework that maps emotions along two axes:

- **Arousal (vertical):** calm → intense — mapped to Spotify's `energy` feature
- **Valence (horizontal):** negative/sad → positive/happy — mapped directly to Spotify's `valence` feature
- **Acousticness** acts as a corrective factor: it disambiguates songs that score similarly on energy/valence but belong to different musical worlds (e.g., acoustic ballad vs. electronic ambient)

| Energy | Valence | Acousticness | Mood Zone | Example |
|--------|---------|-------------|-----------|---------|
| High | High | Low | Joy, Euphoria, Dance | Pop hits, Dance tracks |
| High | Low | Low | Anger, Intensity | Rock, Heavy Metal |
| Low | Low | High | Sadness, Melancholy | Acoustic ballads, Blues |
| Low | High | High | Calm, Serenity | Chill-out, Classical piano |

**References:**
- Russell, J. A. (1980). "A Circumplex Model of Affect." *Journal of Personality and Social Psychology.*
- Spotify Audio Features are validated for music psychology research ([Fuentes-Sánchez et al., 2025](https://journals.sagepub.com/doi/10.1177/10298649261419760))

### 8.2 Implementation

Rather than hard-coding four quadrants, we create a **continuous mood score** from these three features and split each cluster into sub-playlists by sorting along this score. This preserves nuance: songs transition gradually from "low mood" (calm, sad, acoustic) to "high mood" (energetic, happy, electronic).

In [ ]:
# Create mood score from Russell-inspired features
mood_features = ['valence', 'energy', 'acousticness']

weights = {'valence': 1, 'energy': 1, 'acousticness': 1}

scaled_df['mood_score'] = (
    scaled_df['valence'] * weights['valence'] +
    scaled_df['energy'] * weights['energy'] +
    scaled_df['acousticness'] * weights['acousticness']
) / sum(weights.values())

print("Mood score distribution:")
print(scaled_df['mood_score'].describe())

In [ ]:
# Split each cluster into sub-playlists by mood score
target_size = 200  # target songs per playlist

playlist_labels = pd.Series(index=scaled_df.index, dtype=object)

for cluster_id, group in scaled_df.groupby('cluster'):
    # Sort by mood score within each cluster
    group_sorted = group.sort_values('mood_score')
    
    # Calculate number of sub-playlists needed
    n_sub = max(1, round(len(group_sorted) / target_size))
    
    # Split into equal-sized sub-playlists
    subs = np.array_split(group_sorted.index, n_sub)
    
    # Assign playlist labels (e.g., "0_0", "0_1", "1_0", ...)
    for i, sub_index in enumerate(subs):
        playlist_labels.loc[sub_index] = f"{cluster_id}_{i}"

scaled_df['playlist'] = playlist_labels

print(f"Total playlists created: {scaled_df['playlist'].nunique()}")

In [ ]:
# Final playlist size check
playlist_counts = scaled_df['playlist'].value_counts().sort_index()
shortest = playlist_counts.min()
longest = playlist_counts.max()

print(f"Playlist size check (target: 50-250 songs):")
print(f"  Total playlists:    {len(playlist_counts)}")
print(f"  Smallest playlist:  {shortest} songs")
print(f"  Largest playlist:   {longest} songs")
print(f"  Average playlist:   {playlist_counts.mean():.0f} songs")

if shortest >= 50 and longest <= 250:
    print(f"\n  ✅ All playlists within 50-250 range!")
else:
    out_of_range = playlist_counts[(playlist_counts < 50) | (playlist_counts > 250)]
    print(f"\n  ⚠️ {len(out_of_range)} playlists outside 50-250 range")

---
## 9. Playlist Showcase

Let's look at actual songs — do the playlists make sense?

In [ ]:
# Overview: sample songs from each playlist
for p in sorted(scaled_df["playlist"].unique()):
    n_songs = len(scaled_df[scaled_df["playlist"] == p])
    print(f"\n{'='*60}")
    print(f"  Playlist {p}  ({n_songs} songs)")
    print(f"{'='*60}")

    idx = scaled_df[scaled_df["playlist"] == p].index
    subset = songs_df.loc[idx, ["name", "artist"]].head(10)
    for _, row in subset.iterrows():
        name = str(row["name"]).strip()
        artist = str(row["artist"]).strip()
        print(f"  {name} — {artist}")

In [ ]:
# Deep dive: 2 specific playlists
playlist_a = 0
playlist_b = 1

for p in [playlist_a, playlist_b]:
    subset = songs_df[songs_df["cluster"] == p]
    print(f"\n{'='*60}")
    print(f"  Cluster {p}: {len(subset)} songs")
    print(f"{'='*60}")
    print(f"  Avg danceability: {subset['danceability'].mean():.2f}")
    print(f"  Avg energy:       {subset['energy'].mean():.2f}")
    print(f"  Avg valence:      {subset['valence'].mean():.2f}")
    print(f"  Avg acousticness: {subset['acousticness'].mean():.2f}")
    print(f"  Avg tempo:        {subset['tempo'].mean():.0f} BPM")
    print(f"\n  Sample songs:")
    n_sample = min(8, len(subset))
    for _, row in subset[["name", "artist"]].sample(n_sample, random_state=42).iterrows():
        name = str(row["name"]).strip()
        artist = str(row["artist"]).strip()
        print(f"  {name} — {artist}")

### Playlist Assessment

**Playlist 1_1 (Brazilian/Bossa Nova):** Marcos Valle, Seu Jorge, João Gilberto, Wilson Simonal, Tribalistas, Manu Chao — the algorithm grouped Brazilian bossa nova and Latin world music together without knowing what "bossa nova" is. This demonstrates that audio features *can* capture genre-level similarity when songs share distinctive acoustic characteristics.

**Playlist 0_0 (Classical/Ambient):** Chopin, Debussy, Bach, Brian Eno, Max Richter, Aphex Twin — mostly an excellent grouping of classical piano, neo-classical, and ambient electronic. However, it also includes 2–3 death metal tracks (Asphyx, Morgoth) that share low valence scores but sound nothing like Chopin. This shows the algorithm's limitation: **audio features capture mood but miss genre.**

### 9.1 Spotify Links

In [ ]:
# Spotify links for a playlist
show_cluster = 0

print(f"Spotify links — Playlist {show_cluster}:\n")
subset = songs_df[songs_df["cluster"] == show_cluster][["name", "artist", "html"]].head(15)
for _, row in subset.iterrows():
    name = str(row["name"]).strip()
    artist = str(row["artist"]).strip()
    html = str(row["html"]).strip()
    print(f"  {name} — {artist}")
    print(f"  {html}\n")

---
## 10. Experiment: Scaled vs Unscaled

How much does scaling matter?

In [ ]:
# K-Means WITHOUT scaling
kmeans_unscaled = KMeans(n_clusters=chosen_k, random_state=42, n_init="auto")
kmeans_unscaled.fit(features_df)

songs_df["cluster_unscaled"] = kmeans_unscaled.labels_

print("Unscaled cluster sizes:")
print(songs_df["cluster_unscaled"].value_counts().sort_index())
print(f"\nScaled cluster sizes:")
print(songs_df["cluster"].value_counts().sort_index())

**Observation:** Without scaling, clusters are dominated by `tempo` and `loudness` due to their larger range. With scaling, all features contribute equally — producing more balanced, meaningful clusters.

---
## 11. Experiment: MinMaxScaler vs StandardScaler

In [ ]:
# StandardScaler comparison
std_scaler = StandardScaler()
features_std_scaled = std_scaler.fit_transform(features_df)
std_scaled_df = pd.DataFrame(features_std_scaled, columns=features_to_use)

kmeans_std = KMeans(n_clusters=chosen_k, random_state=42, n_init="auto")
kmeans_std.fit(std_scaled_df)

print(f"MinMaxScaler Silhouette:  {silhouette_score(scaled_df[features_to_use], kmeans_final.labels_):.4f}")
print(f"StandardScaler Silhouette: {silhouette_score(std_scaled_df, kmeans_std.labels_):.4f}")

print(f"\nStandardScaler cluster sizes:")
print(pd.Series(kmeans_std.labels_).value_counts().sort_index())

---
## 12. Conclusions

### Q1: How did we create the prototype?

**2-step approach (instructor recommendation):**

**Step 1 — Holistic Clustering (K-Means):**
- 9 audio features: danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo
- Dropped: key, mode, time_signature (categorical), duration_ms (not mood-related)
- MinMaxScaler (0–1) for equal feature contribution
- K-Means with K=8 → 8 broad clusters

**Step 2 — Mood-based Splitting:**
- Created a continuous mood score from valence + energy + acousticness (inspired by Russell's Circumplex Model)
- Within each cluster, sorted songs by mood score and split into sub-playlists of ~200 songs
- Result: 27 playlists of 152–213 songs each ✅

### Q2: Is the prototype effective?

**Yes, with caveats.**

- ✅ All 27 playlists fall within the 50–250 business requirement
- ✅ Songs are grouped by musical similarity (Step 1) and mood (Step 2)
- ✅ Some playlists show remarkable coherence (e.g., Playlist 1_1: all Brazilian bossa nova)

**Limitations:**
- Some playlists mix genres (e.g., Playlist 0_0: classical + death metal — same low valence, completely different sound)
- The mood score is a simplification of complex human music perception

**Verdict:** A solid first prototype. Not production-ready, but demonstrates that ML-based playlist creation is viable and worth developing further.

### Q3: Can audio features identify "similar songs"?

**Partly yes.**

**What they capture well:**
- Energy level (calm vs. intense)
- Mood (happy vs. sad)
- Acoustic vs. electronic character
- Extremes separate clearly (classical will never be grouped with death metal — except when valence scores accidentally align)

**What they miss:**
- Genre (rock vs. jazz vs. classical)
- Language of lyrics
- Era and cultural context
- Subjective "feel" that humans detect instantly

**What would improve results:**
- Genre labels from Spotify's API
- Lyrics analysis (NLP)
- User listening behavior (collaborative filtering)

### Q4: Is K-Means a good method for playlists?

**For prototyping: yes. For production: needs enhancement.**

**Pros:**
- Simple, fast, easy to explain to non-technical stakeholders
- Scalable — runs in seconds even for 5,000+ songs
- Good starting point for iteration

**Cons:**
- Assumes spherical clusters of similar size
- Sensitive to initial centroid placement (mitigated by `n_init`)
- Requires you to choose K manually
- Doesn't capture complex cluster shapes

**Alternatives worth exploring:**
- **DBSCAN:** No K needed, finds outliers automatically
- **Hierarchical Clustering:** Shows relationships between clusters via dendrogram
- **Gaussian Mixture Models:** More flexible cluster shapes, soft assignments

### Q5: Next Steps

1. **A/B test** ML playlists vs. expert-curated with real Moosic subscribers
2. **Enrich data** with genre labels and lyrics analysis
3. **User feedback loop** — let subscribers rate playlists; every thumbs up/down improves the algorithm
4. **Try alternative algorithms** (DBSCAN, hierarchical clustering)
5. **Hybrid approach:** Algorithm creates first draft → music experts add the finishing touch

**Our recommendation:** Use ML to scale the automation. Keep the human touch that makes Moosic special.

---
*Moosic Case Study — WBS Coding School Data Science Bootcamp*